In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


self-contained Python script that generates an offline RL dataset for discrete RIS control (6G). It simulates BS–RIS–UE channels, builds a discrete RIS codebook, runs a behavior policy (ε-greedy over a simple heuristic), computes rates/SINR with MRT precoding, and logs clean tuples
(
𝑠
,
𝑎
,
𝑟
,
𝑠
′
)
(s,a,r,s
′
) for DQN/CQL/etc.

In [ ]:
# ==============================
# 1. Install dependencies
# ==============================
!pip install pandas pyarrow torch --quiet


The data generation block builds a wireless environment where a base station communicates with users through a reconfigurable intelligent surface (RIS). First, it defines a discrete action space (a codebook of RIS reflection patterns). At each step, the environment randomly picks an action (i.e., a RIS phase configuration), computes the resulting end-to-end channels (direct BS→UE plus RIS-assisted paths), applies a simple precoder at the BS, and evaluates the SINR for each user. A reward is then calculated using a proportional fairness metric (sum of log-rates across users). The code records the current state (placeholder features), the chosen action, the reward, and the next state into a dataset. Repeating this process over many episodes produces a table of
(
𝑠
,
𝑎
,
𝑟
,
𝑠
′
)
(s,a,r,s
′
) transitions that can be saved (CSV/Parquet) and used to train reinforcement learning agents offline.

In [ ]:
# ==============================
# 2. Dataset Generator
# ==============================
import os, json
import numpy as np
import pandas as pd

# ---- Helpers ----
def db2lin(x_db): return 10.0**(x_db/10.0)
def lin2db(x): return 10.0*np.log10(max(x,1e-30))
def complex_randn(shape): return (np.random.randn(*shape) + 1j*np.random.randn(*shape))/np.sqrt(2)

# ---- RIS Codebook ----
class RISCodebook:
    def __init__(self, N=64, G=8, K=8):
        assert N % G == 0
        self.N, self.G, self.K = N, G, K
        self.group_size = N // G
        self.phase_options = np.linspace(0, 2*np.pi, num=K, endpoint=False)

    def size(self):
        return self.K**self.G

    def action_to_diag(self, a:int):
        # Convert integer a to group-phase assignment
        digits = []
        x = a
        for _ in range(self.G):
            digits.append(x % self.K)
            x //= self.K
        digits = digits[::-1]
        phases = self.phase_options[digits]
        phase_vec = np.repeat(phases, self.group_size)
        return np.diag(np.exp(1j*phase_vec))

# ---- Channel + SINR ----
def build_effective(H_BR, h_RU, h_BU, Phi):
    GU = []
    for u in range(len(h_RU)):
        g = h_BU[u].conj().T + h_RU[u].conj().T @ Phi @ H_BR
        GU.append(g.reshape(1,-1))
    return GU

def mrt_precoder(GU, Ptx):
    U = len(GU)
    if U == 0: return []
    W = []
    for g in GU:
        v = g.conj().T
        w = v / (np.linalg.norm(v) + 1e-12) * np.sqrt(Ptx / U)
        W.append(w)
    return W

def compute_sinr(GU, W, noise):
    U = len(GU); out = np.zeros(U)
    for u in range(U):
        g = GU[u]
        num = abs(g @ W[u])**2
        denom = noise + sum(abs(g @ W[v])**2 for v in range(U) if v != u)
        out[u] = float(num / denom)
    return out

# ---- Reward ----
def pf_reward(sinr):
    return float(np.sum(np.log2(1+sinr)))

# ---- Env ----
class SimpleRISEnv:
    def __init__(self, N=64, M=8, U=6, G=8, K=8):
        self.N, self.M, self.U = N, M, U
        self.cb = RISCodebook(N, G, K)
        self.noise = db2lin(-100/10)   # ~ -100 dBm noise
        self.Ptx = db2lin(30 - 30)     # 1 W transmit power

    def extract_state(self, GU, sinr):
        feats = []
        # Effective channel strength per user
        feats += [np.linalg.norm(g)**2 for g in GU]
        # Current SINR per user
        feats += list(sinr)
        return np.array(feats, dtype=np.float32)

    def reset(self):
        self.H_BR = complex_randn((self.N, self.M))
        self.h_RU = [complex_randn((self.N,)) for _ in range(self.U)]
        self.h_BU = [complex_randn((self.M,)) for _ in range(self.U)]
        # Start with identity RIS (no phase control)
        Phi = np.eye(self.N)
        GU = build_effective(self.H_BR, self.h_RU, self.h_BU, Phi)
        W = mrt_precoder(GU, self.Ptx)
        sinr = compute_sinr(GU, W, self.noise)
        self.state = self.extract_state(GU, sinr)
        return self.state

    def step(self, a):
        Phi = self.cb.action_to_diag(a)
        GU = build_effective(self.H_BR, self.h_RU, self.h_BU, Phi)
        W = mrt_precoder(GU, self.Ptx)
        sinr = compute_sinr(GU, W, self.noise)
        r = pf_reward(sinr)
        s_next = self.extract_state(GU, sinr)
        return s_next, r, False, {"mean_sinr": np.mean(sinr)}

# ---- Generate dataset ----
def generate_dataset(episodes=20, steps=200, outdir="out"):
    env = SimpleRISEnv()
    rows = []
    for ep in range(episodes):
        s = env.reset()
        for t in range(steps):
            a = np.random.randint(0, env.cb.size())  # random policy
            s_next, r, d, info = env.step(a)
            # mark last step in the episode as done
            done_flag = (t == steps - 1)
            rows.append({
                "episode": ep, "t": t,
                "state": json.dumps(s.tolist()),
                "action": a, "reward": r,
                "next_state": json.dumps(s_next.tolist()),
                "done": done_flag,
                "mean_sinr": info["mean_sinr"]
            })
            s = s_next
    df = pd.DataFrame(rows)
    os.makedirs(outdir, exist_ok=True)
    df.to_csv(os.path.join(outdir, "ris_dataset.csv"), index=False)
    print("Saved dataset:", df.shape)
    return df

# Example run
df = generate_dataset()
df.to_csv("/content/gdrive/MyDrive/ris_dataset.csv", index=False)
print(df.head(15))


/tmp/ipython-input-3927941608.py:60: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  out[u] = float(num / denom)


Saved dataset: (4000, 8)
    episode   t                                              state    action  \
0         0   0  [141.4363555908203, 344.2704772949219, 607.625...   1556274   
1         0   1  [347.1873474121094, 137.4445037841797, 432.113...   7594989   
2         0   2  [453.2982482910156, 566.0628051757812, 417.591...   1941769   
3         0   3  [435.334716796875, 418.2289733886719, 704.4493...  14268763   
4         0   4  [293.1031188964844, 538.6586303710938, 197.572...  12289467   
5         0   5  [300.7584533691406, 343.6442565917969, 329.285...   9045524   
6         0   6  [295.57843017578125, 185.08673095703125, 470.8...   2679739   
7         0   7  [263.3471374511719, 654.9554443359375, 248.180...   8999792   
8         0   8  [174.31103515625, 571.1026000976562, 568.54400...  12989451   
9         0   9  [309.9609069824219, 462.42364501953125, 449.35...   4905690   
10        0  10  [228.05284118652344, 490.9288024902344, 619.14...   5330354   
11        0  11

# reminder: need colab pro for increased RAM

In [ ]:
# ==============================
# 3. Offline DQN Training
# ==============================
import torch
import torch.nn as nn
import torch.optim as optim
import json

# ---- Dataset loader ----
data=pd.read_csv("/content/gdrive/MyDrive/ris_dataset.csv")
states=np.vstack(data["state"].apply(json.loads).values)
next_states=np.vstack(data["next_state"].apply(json.loads).values)
actions=data["action"].values
rewards=data["reward"].values.astype(np.float32)
dones=data["done"].values.astype(np.float32)

state_dim=states.shape[1]
n_actions=int(data["action"].max()+1)

print("State dim:",state_dim,"Actions:",n_actions)

# ---- DQN Network ----
class DQN(nn.Module):
    def __init__(self,state_dim,n_actions):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(state_dim,128),nn.ReLU(),
            nn.Linear(128,128),nn.ReLU(),
            nn.Linear(128,n_actions)
        )
    def forward(self,x): return self.net(x)

q_net=DQN(state_dim,n_actions)
target_net=DQN(state_dim,n_actions)
target_net.load_state_dict(q_net.state_dict())
opt=optim.Adam(q_net.parameters(),lr=1e-3)
loss_fn=nn.MSELoss()

# ---- Training loop ----
gamma=0.99
batch_size=64
epochs=10

dataset_size=len(states)
for epoch in range(epochs):
    perm=np.random.permutation(dataset_size)
    losses=[]
    for i in range(0,dataset_size,batch_size):
        idx=perm[i:i+batch_size]
        s=torch.tensor(states[idx],dtype=torch.float32)
        ns=torch.tensor(next_states[idx],dtype=torch.float32)
        a=torch.tensor(actions[idx],dtype=torch.long)
        r=torch.tensor(rewards[idx],dtype=torch.float32)
        d=torch.tensor(dones[idx],dtype=torch.float32)

        qvals=q_net(s).gather(1,a.unsqueeze(1)).squeeze(1)
        with torch.no_grad():
            max_next=torch.max(target_net(ns),1)[0]
            y=r+gamma*max_next*(1-d)
        loss=loss_fn(qvals,y)

        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())

    target_net.load_state_dict(q_net.state_dict())
    print(f"Epoch {epoch+1}, loss={np.mean(losses):.4f}")


State dim: 12 Actions: 16774149


In [ ]:
# ==============================
# 4. Quick Evaluation
# ==============================
# Take first few states and see what actions Q-net prefers
sample=torch.tensor(states[:5],dtype=torch.float32)
with torch.no_grad():
    qvals=q_net(sample)
print("Q-values (first 5 states):")
print(qvals)
print("Chosen actions:",qvals.argmax(1).numpy())
